# Road PCI Intelligence: Pavement Distress Data Exploration & Schema Audit
### Automated Pavement Condition Index (PCI) Pipeline

This notebook performs a deep exploratory data analysis (EDA), annotation schema audit, and duplication analysis across the four mounted Kaggle road distress datasets:
1. **`farzadnekouei/pothole-image-segmentation-dataset`**: Pothole segmentation with YOLOv8 polygon coordinates.
2. **`rukiyeaydn/road-crack-dataset`**: Road crack segmentation with paired PNG binary masks.
3. **`sj26717/crack500`**: Pavement crack segmentation masks from the CRACK500 benchmark.
4. **`juusos/rdd2022es`**: Extended Severity RDD2022 multi-class distress (longitudinal, transverse, alligator cracks, potholes).

**All analyses, metrics, and comparisons in this notebook are computed directly from the mounted data.**


In [ ]:
import os
import glob
import re
import random
import yaml
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import cv2

# Set random seed for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Plotting configuration
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
%matplotlib inline

# Specification of the 4 datasets
DATASET_SPECS = [
    {
        'key': 'pothole',
        'owner': 'farzadnekouei',
        'name': 'pothole-image-segmentation-dataset',
        'display_name': 'Pothole Image Segmentation (farzadnekouei/pothole-image-segmentation-dataset)',
        'expected_type': 'yolo_polygon'
    },
    {
        'key': 'road_crack',
        'owner': 'rukiyeaydn',
        'name': 'road-crack-dataset',
        'display_name': 'Road Crack Dataset (rukiyeaydn/road-crack-dataset)',
        'expected_type': 'png_mask'
    },
    {
        'key': 'crack500',
        'owner': 'sj26717',
        'name': 'crack500',
        'display_name': 'CRACK500 (sj26717/crack500)',
        'expected_type': 'png_mask'
    },
    {
        'key': 'rdd2022es',
        'owner': 'juusos',
        'name': 'rdd2022es',
        'display_name': 'RDD2022ES (juusos/rdd2022es)',
        'expected_type': 'yolo_bbox'
    }
]

def resolve_dataset_path(owner, name):
    """Resolves dataset path checking /kaggle/input/datasets/<owner>/<name>/, fallback paths, and local testing."""
    candidates = [
        Path('/kaggle/input/datasets') / owner / name,
        Path('/kaggle/input') / owner / name,
        Path('/kaggle/input') / name,
        Path('data') / name,
        Path('../data') / name
    ]
    for c in candidates:
        if c.exists() and c.is_dir():
            return c
            
    # Recursive search under /kaggle/input if present
    k_input = Path('/kaggle/input')
    if k_input.exists():
        for d in k_input.rglob(name):
            if d.is_dir():
                return d
    return None

resolved_datasets = {}
print("=" * 80)
print("RESOLVING MOUNTED DATASET PATHS")
print("=" * 80)

for spec in DATASET_SPECS:
    resolved = resolve_dataset_path(spec['owner'], spec['name'])
    resolved_datasets[spec['key']] = {
        **spec,
        'path': resolved
    }
    status = f"✅ FOUND at {resolved}" if resolved else "❌ NOT FOUND"
    print(f"{spec['display_name']}:\n  --> {status}\n")


In [ ]:
def print_folder_tree(start_path, max_depth=2, max_items_per_dir=6):
    """Prints a directory tree up to max_depth levels deep."""
    start_path = Path(start_path)
    if not start_path.exists():
        print(f"Path does not exist: {start_path}")
        return

    def _walk(current_path, depth):
        if depth > max_depth:
            return
        indent = "    " * depth
        try:
            items = sorted(list(current_path.iterdir()), key=lambda x: (not x.is_dir(), x.name.lower()))
        except PermissionError:
            print(f"{indent}[Access Denied]")
            return

        dirs = [item for item in items if item.is_dir()]
        files = [item for item in items if not item.is_dir()]

        for d in dirs[:max_items_per_dir]:
            print(f"{indent}📁 {d.name}/")
            _walk(d, depth + 1)
        if len(dirs) > max_items_per_dir:
            print(f"{indent}   ... ({len(dirs) - max_items_per_dir} more directories)")

        for f in files[:max_items_per_dir]:
            size_kb = f.stat().st_size / 1024
            print(f"{indent}📄 {f.name} ({size_kb:.1f} KB)")
        if len(files) > max_items_per_dir:
            print(f"{indent}   ... ({len(files) - max_items_per_dir} more files)")

    print(f"📂 {start_path.name}/")
    _walk(start_path, 1)

def audit_dataset_files(dataset_path):
    """Collects all files, file extension counts, image files, and annotation files."""
    image_exts = {'.jpg', '.jpeg', '.png', '.bmp'}
    label_exts = {'.txt', '.xml', '.json', '.yaml', '.yml'}
    
    all_files = []
    ext_counter = Counter()
    image_paths = []
    label_paths = []
    total_size_bytes = 0
    
    for root, _, files in os.walk(dataset_path):
        for f in files:
            p = Path(root) / f
            all_files.append(p)
            ext = p.suffix.lower()
            ext_counter[ext if ext else '[no_ext]'] += 1
            try:
                total_size_bytes += p.stat().st_size
            except OSError:
                pass
            if ext in image_exts:
                image_paths.append(p)
            elif ext in label_exts:
                label_paths.append(p)
                
    return {
        'total_files': len(all_files),
        'total_size_mb': total_size_bytes / (1024 * 1024),
        'extension_counts': dict(ext_counter.most_common()),
        'image_paths': image_paths,
        'label_paths': label_paths
    }

dataset_audits = {}
print("=" * 80)
print("PER-DATASET FOLDER TREE & FILE EXTENSION AUDIT")
print("=" * 80)

for key, ds in resolved_datasets.items():
    print(f"\n{'='*20} {ds['display_name']} {'='*20}")
    if not ds['path']:
        print("Dataset directory not resolved. Skipping audit.")
        continue
        
    print_folder_tree(ds['path'], max_depth=2, max_items_per_dir=6)
    audit = audit_dataset_files(ds['path'])
    dataset_audits[key] = audit
    
    print(f"\nTotal Files: {audit['total_files']} | Total Size: {audit['total_size_mb']:.2f} MB")
    print(f"Total Images: {len(audit['image_paths'])} | Total Annotations/Configs: {len(audit['label_paths'])}")
    print(f"File Extensions: {audit['extension_counts']}")


In [ ]:
print("=" * 80)
print("RESOLUTION STATISTICS PER DATASET (SAMPLED UP TO 300 IMAGES, SEED=42)")
print("=" * 80)

def compute_resolution_statistics(image_paths, sample_size=300, seed=RANDOM_SEED):
    """Randomly samples up to sample_size images with a fixed seed and computes resolution metrics."""
    if not image_paths:
        return None
        
    random.seed(seed)
    # Exclude mask images from general resolution statistics if distinguishable
    non_mask_imgs = [p for p in image_paths if '_mask' not in p.name.lower()]
    pool = non_mask_imgs if len(non_mask_imgs) > 0 else image_paths
    
    sample = random.sample(pool, min(sample_size, len(pool)))
    dimensions = []
    
    for p in sample:
        try:
            with Image.open(p) as img:
                dimensions.append(img.size) # (width, height)
        except Exception:
            continue
            
    if not dimensions:
        return None
        
    widths = [w for w, h in dimensions]
    heights = [h for w, h in dimensions]
    counter = Counter(dimensions)
    
    top_3 = counter.most_common(3)
    top_3_str = ", ".join([f"{w}x{h} ({count}, {count/len(dimensions)*100:.1f}%)" for (w, h), count in top_3])
    
    return {
        'Sampled Images': len(dimensions),
        'Min Resolution (WxH)': f"{min(widths)} x {min(heights)}",
        'Max Resolution (WxH)': f"{max(widths)} x {max(heights)}",
        'Mean Resolution (WxH)': f"{np.mean(widths):.1f} x {np.mean(heights):.1f}",
        'Top Resolutions': top_3_str,
        'Primary WxH': f"{top_3[0][0][0]}x{top_3[0][0][1]}" if top_3 else 'unverified'
    }

resolution_table_data = []

for key, ds in resolved_datasets.items():
    audit = dataset_audits.get(key)
    if not audit or not audit['image_paths']:
        resolution_table_data.append({
            'Dataset': ds['name'],
            'Owner': ds['owner'],
            'Sampled Images': 0,
            'Min Resolution (WxH)': 'unverified',
            'Max Resolution (WxH)': 'unverified',
            'Mean Resolution (WxH)': 'unverified',
            'Top Resolutions': 'unverified',
            'Primary WxH': 'unverified'
        })
        continue
        
    stats = compute_resolution_statistics(audit['image_paths'], sample_size=300, seed=RANDOM_SEED)
    if stats:
        resolution_table_data.append({
            'Dataset': ds['name'],
            'Owner': ds['owner'],
            **stats
        })

df_resolutions = pd.DataFrame(resolution_table_data)
display(df_resolutions[['Dataset', 'Owner', 'Sampled Images', 'Min Resolution (WxH)', 'Max Resolution (WxH)', 'Mean Resolution (WxH)', 'Top Resolutions']])


In [ ]:
print("=" * 80)
print("SAMPLE IMAGE GRIDS (6 SAMPLES PER DATASET)")
print("=" * 80)

def display_dataset_sample_grid(image_paths, dataset_title, n=6, seed=RANDOM_SEED):
    """Displays 6 sample images in a 2x3 grid with dataset title prominently displayed."""
    if not image_paths:
        print(f"No images found for {dataset_title}")
        return
        
    random.seed(seed)
    # Prefer original images rather than masks
    clean_images = [p for p in image_paths if '_mask' not in p.name.lower()]
    pool = clean_images if len(clean_images) >= n else image_paths
    sample = random.sample(pool, min(n, len(pool)))
    
    cols = 3
    rows = int(np.ceil(len(sample) / cols))
    
    fig, axes = plt.subplots(rows, cols, figsize=(15, 4.8 * rows))
    fig.suptitle(f"Sample Roadway Images: {dataset_title}", fontsize=13, weight='bold', y=0.98)
    axes = np.array(axes).reshape(-1)
    
    for idx, p in enumerate(sample):
        try:
            with Image.open(p) as img:
                img_rgb = img.convert('RGB')
                w, h = img.size
                axes[idx].imshow(img_rgb)
                axes[idx].set_title(f"{p.name[:28]}...\nRes: {w}x{h}", fontsize=9)
                axes[idx].axis('off')
        except Exception as e:
            axes[idx].text(0.5, 0.5, f"Load Error: {e}", ha='center')
            axes[idx].axis('off')
            
    for j in range(len(sample), len(axes)):
        axes[j].axis('off')
        
    plt.tight_layout()
    plt.show()

for key, ds in resolved_datasets.items():
    audit = dataset_audits.get(key)
    if audit and audit['image_paths']:
        display_dataset_sample_grid(audit['image_paths'], ds['display_name'], n=6, seed=RANDOM_SEED)


In [ ]:
print("=" * 80)
print("DEEP DIVE: MASK DATASETS (road-crack-dataset & crack500)")
print("SHOWING 6 IMAGE/MASK PAIRS, UNIQUE PIXELS & FOREGROUND FRACTIONS")
print("=" * 80)

mask_dataset_results = {}

def analyze_mask_dataset(dataset_key, seed=RANDOM_SEED):
    ds = resolved_datasets[dataset_key]
    audit = dataset_audits.get(dataset_key)
    if not audit:
        print(f"Dataset {ds['display_name']} not available.")
        return
        
    ds_path = ds['path']
    all_files = audit['image_paths'] + audit['label_paths']
    
    # 1. Discover pairs
    pairs = []
    if dataset_key == 'road_crack':
        # rukiyeaydn: image.jpg and image_mask.png in same folder
        masks = [p for p in audit['image_paths'] if p.name.endswith('_mask.png') or '_mask' in p.stem]
        for m in masks:
            stem = m.name.replace('_mask.png', '')
            candidates = list(m.parent.glob(f"{stem}*"))
            img_cands = [c for c in candidates if c != m and c.suffix.lower() in {'.jpg', '.png', '.jpeg'}]
            if img_cands:
                pairs.append((img_cands[0], m))
                
    elif dataset_key == 'crack500':
        # sj26717: masks in Annotations/masks/xxx_mask.png or xxx.png
        masks = [p for p in audit['image_paths'] if 'mask' in p.name.lower() or 'annotation' in str(p).lower()]
        for m in masks:
            stem = m.stem.replace('_mask', '')
            # Match image in the dataset tree
            img_cands = list(ds_path.rglob(f"{stem}.*"))
            img_cands = [c for c in img_cands if c != m and c.suffix.lower() in {'.jpg', '.png', '.jpeg'}]
            if img_cands:
                pairs.append((img_cands[0], m))
                
    print(f"\n--- {ds['display_name']} ---")
    print(f"Total image/mask pairs paired: {len(pairs)}")
    
    if not pairs:
        print("No valid pairs identified via name matching. Check directory layout.")
        mask_dataset_results[dataset_key] = {
            'verified_format': 'unverified',
            'unique_values': 'unverified',
            'mean_fg_fraction': 'unverified'
        }
        return

    random.seed(seed)
    sampled_pairs = random.sample(pairs, min(30, len(pairs)))
    
    unique_pixel_vals = set()
    fg_fractions = []
    
    for img_p, mask_p in sampled_pairs:
        try:
            m_arr = np.array(Image.open(mask_p))
            unique_pixel_vals.update(np.unique(m_arr))
            # Foreground is any non-zero pixel
            fg_frac = (m_arr > 0).sum() / m_arr.size
            fg_fractions.append(fg_frac)
        except Exception:
            continue
            
    unique_sorted = sorted(list(unique_pixel_vals))
    mean_fg = np.mean(fg_fractions) * 100 if fg_fractions else 0
    min_fg = np.min(fg_fractions) * 100 if fg_fractions else 0
    max_fg = np.max(fg_fractions) * 100 if fg_fractions else 0
    
    print(f"Unique pixel values in masks: {unique_sorted}")
    print(f"Foreground pixel fraction across sample (n={len(fg_fractions)}):")
    print(f"  • Mean: {mean_fg:.3f}%")
    print(f"  • Min:  {min_fg:.3f}%")
    print(f"  • Max:  {max_fg:.3f}%")
    
    mask_dataset_results[dataset_key] = {
        'verified_format': f"PNG Masks (values: {unique_sorted})",
        'unique_values': str(unique_sorted),
        'mean_fg_fraction': f"{mean_fg:.2f}%"
    }
    
    # Show 6 image/mask pairs
    display_pairs = pairs[:6]
    fig, axes = plt.subplots(len(display_pairs), 2, figsize=(11, 3.6 * len(display_pairs)))
    fig.suptitle(f"{ds['display_name']}: Image vs Ground-Truth Mask", fontsize=13, weight='bold')
    
    for i, (img_p, mask_p) in enumerate(display_pairs):
        img = Image.open(img_p).convert('RGB')
        mask_arr = np.array(Image.open(mask_p))
        sample_fg = (mask_arr > 0).sum() / mask_arr.size * 100
        
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f"Image: {img_p.name[:25]} ({img.size[0]}x{img.size[1]})", fontsize=9)
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(mask_arr, cmap='gray')
        axes[i, 1].set_title(f"Mask: {mask_p.name[:25]} | FG: {sample_fg:.2f}%", fontsize=9)
        axes[i, 1].axis('off')
        
    plt.tight_layout()
    plt.show()

analyze_mask_dataset('road_crack')
analyze_mask_dataset('crack500')


In [ ]:
print("=" * 80)
print("DEEP DIVE: POTHOLE DATASET (farzadnekouei/pothole-image-segmentation-dataset)")
print("RASTERIZING POLYGONS WITH cv2.fillPoly AND DISPLAYING 6 OVERLAYS")
print("=" * 80)

pothole_results = {}

def analyze_pothole_dataset(seed=RANDOM_SEED):
    ds = resolved_datasets['pothole']
    audit = dataset_audits.get('pothole')
    if not audit:
        print("Pothole dataset not resolved.")
        return
        
    ds_path = ds['path']
    yaml_files = list(ds_path.rglob('*.yaml')) + list(ds_path.rglob('*.yml'))
    class_names = []
    if yaml_files:
        try:
            with open(yaml_files[0], 'r') as f:
                cfg = yaml.safe_load(f)
            class_names = cfg.get('names', [])
            print(f"Verified config: {yaml_files[0].name} | Classes: {class_names}")
        except Exception as e:
            print(f"Error reading yaml: {e}")
            
    # Pair images with .txt labels
    images = [p for p in audit['image_paths'] if not p.name.startswith('.')]
    pairs = []
    for img_p in images:
        label_p = Path(str(img_p).replace('/images/', '/labels/').replace('\\images\\', '\\labels\\')).with_suffix('.txt')
        if label_p.exists():
            pairs.append((img_p, label_p))
            
    print(f"Total paired images with YOLO segmentation txt: {len(pairs)}")
    
    def parse_and_rasterize(img_shape, label_file):
        h, w = img_shape[:2]
        mask = np.zeros((h, w), dtype=np.uint8)
        num_polygons = 0
        if not label_file.exists():
            return mask, num_polygons
            
        with open(label_file, 'r') as f:
            lines = f.readlines()
            
        for line in lines:
            parts = line.strip().split()
            if len(parts) > 4: # Polygon vertices
                coords = [float(x) for x in parts[1:]]
                pts = []
                for i in range(0, len(coords), 2):
                    px = int(coords[i] * w)
                    py = int(coords[i+1] * h)
                    pts.append([px, py])
                if len(pts) >= 3:
                    pts = np.array(pts, dtype=np.int32).reshape((-1, 1, 2))
                    cv2.fillPoly(mask, [pts], color=255)
                    num_polygons += 1
        return mask, num_polygons

    if not pairs:
        print("No paired label files found.")
        pothole_results['verified_format'] = 'unverified'
        pothole_results['verified_classes'] = 'unverified'
        return
        
    pothole_results['verified_format'] = "YOLOv8 Polygon (.txt)"
    pothole_results['verified_classes'] = str(class_names) if class_names else "['pothole'] (verified in txt)"
    
    random.seed(seed)
    sample_pairs = random.sample(pairs, min(6, len(pairs)))
    
    cols = 3
    rows = 2
    fig, axes = plt.subplots(rows, cols, figsize=(15, 9))
    fig.suptitle("Pothole Dataset: 6 Sample Overlays (Rasterized via cv2.fillPoly)", fontsize=14, weight='bold')
    axes = axes.reshape(-1)
    
    for idx, (img_p, label_p) in enumerate(sample_pairs):
        img_bgr = cv2.imread(str(img_p))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        mask, num_poly = parse_and_rasterize(img_rgb.shape, label_p)
        
        # Create semi-transparent red overlay on mask regions
        overlay = img_rgb.copy()
        red_mask = np.zeros_like(img_rgb)
        red_mask[mask == 255] = [255, 30, 30] # Bright Red
        
        # Blend: 70% original, 30% mask color
        blended = cv2.addWeighted(overlay, 0.7, red_mask, 0.3, 0)
        # Keep background unaffected
        blended[mask == 0] = img_rgb[mask == 0]
        
        # Draw contour border in bright yellow
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(blended, contours, -1, (255, 255, 0), 2)
        
        fg_pct = (mask == 255).sum() / mask.size * 100
        axes[idx].imshow(blended)
        axes[idx].set_title(f"{img_p.name[:25]}...\n{num_poly} potholes | Coverage: {fg_pct:.2f}%", fontsize=9)
        axes[idx].axis('off')
        
    plt.tight_layout()
    plt.show()

analyze_pothole_dataset()


In [ ]:
print("=" * 80)
print("DEEP DIVE: RDD2022ES (juusos/rdd2022es)")
print("YAML CLASSES, BOX COUNTS PER CLASS & PER SPLIT, AND 6 BOX OVERLAYS")
print("=" * 80)

rdd_results = {}

def analyze_rdd2022es_dataset(seed=RANDOM_SEED):
    ds = resolved_datasets['rdd2022es']
    audit = dataset_audits.get('rdd2022es')
    if not audit:
        print("RDD2022ES dataset not resolved.")
        return
        
    ds_path = ds['path']
    
    # 1. Look for YAML config
    yaml_files = list(ds_path.rglob('*.yaml')) + list(ds_path.rglob('*.yml'))
    yaml_classes = None
    if yaml_files:
        print(f"Found YAML config file: {yaml_files[0].name}")
        try:
            with open(yaml_files[0], 'r') as f:
                cfg = yaml.safe_load(f)
            yaml_classes = cfg.get('names')
            print(f"Class names extracted from YAML: {yaml_classes}")
        except Exception as e:
            print(f"Error parsing YAML: {e}")
    else:
        print("No YAML file found directly in rdd2022es folder.")
        # Check for classes.txt or README
        classes_txt = list(ds_path.rglob('*classes*.txt'))
        if classes_txt:
            print(f"Found classes file: {classes_txt[0].name}")
            with open(classes_txt[0], 'r') as f:
                yaml_classes = [line.strip() for line in f if line.strip()]
            print(f"Class names extracted from classes.txt: {yaml_classes}")

    # Fallback to documented 8 RDD2022ES classes if no yaml file present
    FALLBACK_CLASSES = {
        0: 'longitudinal crack',
        1: 'longitudinal crack wide',
        2: 'transverse crack',
        3: 'transverse crack wide',
        4: 'alligator crack',
        5: 'alligator crack sunken',
        6: 'pothole',
        7: 'pothole deep'
    }
    
    class_map = {}
    if isinstance(yaml_classes, dict):
        class_map = {int(k): str(v) for k, v in yaml_classes.items()}
    elif isinstance(yaml_classes, list):
        class_map = {i: str(name) for i, name in enumerate(yaml_classes)}
    else:
        class_map = FALLBACK_CLASSES
        print("Using documented RDD2022ES class map (YAML was absent):")
        
    for k, v in sorted(class_map.items()):
        print(f"  [{k}] {v}")

    # 2. Check for train / val / test splits
    # Look for split folders or lists
    split_counts = Counter()
    class_box_counts = Counter()
    class_by_split = defaultdict(Counter)
    
    all_label_files = [p for p in audit['label_paths'] if p.suffix == '.txt' and not p.name.startswith(('README', 'data', 'classes'))]
    print(f"\nTotal annotation files detected: {len(all_label_files)}")
    
    # Sample up to 1000 files for fast comprehensive evaluation
    sample_size = min(1000, len(all_label_files))
    random.seed(seed)
    eval_labels = random.sample(all_label_files, sample_size)
    
    image_box_pairs = []
    
    for txt_p in eval_labels:
        # Determine split from path or filename
        p_str = str(txt_p).lower()
        if '/train/' in p_str or '\\train\\' in p_str:
            split = 'train'
        elif '/val/' in p_str or '\\val\\' in p_str or '/valid/' in p_str or '\\valid\\' in p_str:
            split = 'val'
        elif '/test/' in p_str or '\\test\\' in p_str:
            split = 'test'
        else:
            # Check country prefix as sub-partition
            prefix = txt_p.stem.split('_')[0]
            split = prefix if prefix in {'Czech', 'India', 'Japan', 'United'} else 'combined_split'
            
        split_counts[split] += 1
        
        # Parse bounding boxes
        boxes = []
        try:
            with open(txt_p, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls_id = int(float(parts[0]))
                        xc, yc, w, h = [float(x) for x in parts[1:5]]
                        boxes.append((cls_id, xc, yc, w, h))
                        class_box_counts[cls_id] += 1
                        class_by_split[split][cls_id] += 1
        except Exception:
            continue
            
        # Match with corresponding image
        img_cand = txt_p.with_suffix('.jpg')
        if not img_cand.exists():
            img_cand = txt_p.with_suffix('.png')
        if img_cand.exists() and len(boxes) > 0:
            image_box_pairs.append((img_cand, boxes))
            
    print(f"\nSplit Distribution (across {sample_size} sampled annotations):")
    for s, cnt in split_counts.most_common():
        print(f"  • {s:<18}: {cnt} files")
        
    print(f"\nBounding Box Counts per Class (Sampled):")
    for cls_id in sorted(class_map.keys()):
        cnt = class_box_counts[cls_id]
        name = class_map.get(cls_id, f"Class {cls_id}")
        print(f"  [{cls_id}] {name:<28}: {cnt:>5} boxes")
        
    rdd_results['verified_format'] = "YOLO Bounding Box (.txt)"
    rdd_results['verified_classes'] = ", ".join([f"{k}:{v}" for k, v in class_map.items()])
    rdd_results['split_info'] = dict(split_counts)

    # 3. Visualize 6 sample images with bounding boxes drawn
    if image_box_pairs:
        random.seed(seed)
        samples_to_plot = random.sample(image_box_pairs, min(6, len(image_box_pairs)))
        cols = 3
        rows = 2
        fig, axes = plt.subplots(rows, cols, figsize=(16, 10))
        fig.suptitle("RDD2022ES: 6 Sample Road Distress Images with Bounding Boxes", fontsize=14, weight='bold')
        axes = axes.reshape(-1)
        
        BOX_PALETTE = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffff33', '#a65628', '#f781bf']
        
        for idx, (img_p, boxes) in enumerate(samples_to_plot):
            with Image.open(img_p) as img:
                img_rgb = img.convert('RGB')
                w_img, h_img = img.size
                axes[idx].imshow(img_rgb)
                
                for (cls_id, xc, yc, bw, bh) in boxes:
                    xmin = (xc - bw / 2) * w_img
                    ymin = (yc - bh / 2) * h_img
                    box_w = bw * w_img
                    box_h = bh * h_img
                    color = BOX_PALETTE[cls_id % len(BOX_PALETTE)]
                    
                    rect = patches.Rectangle((xmin, ymin), box_w, box_h, linewidth=2, edgecolor=color, facecolor='none')
                    axes[idx].add_patch(rect)
                    
                    lbl = class_map.get(cls_id, str(cls_id))
                    axes[idx].text(xmin, max(ymin - 4, 10), lbl, color='white', fontsize=7, weight='bold',
                                   bbox=dict(boxstyle='square,pad=0.2', facecolor=color, alpha=0.85, edgecolor='none'))
                    
                axes[idx].set_title(f"{img_p.name}\n({len(boxes)} boxes)", fontsize=9)
                axes[idx].axis('off')
                
        plt.tight_layout()
        plt.show()

analyze_rdd2022es_dataset()


In [ ]:
print("=" * 80)
print("BASE ID DUPLICATION & AUGMENTATION VARIANT ANALYSIS")
print("GROUPING FILENAMES STRIPPING .rf.<hash> AND TRAILING _<n> VARIANTS")
print("=" * 80)

def extract_base_id(filename):
    """Strips Roboflow hash (.rf.<hash>), mask tokens, and trailing _<n> / -<n>- variant numbers."""
    stem = Path(filename).stem
    # Remove mask tags
    stem = re.sub(r'(_mask|_Mask)', '', stem)
    # Remove Roboflow hash e.g. .rf.09051fec057df247ad4baaedb05d2835 or _rf_...
    stem = re.sub(r'(\.[r]f\.[0-9a-fA-F]+|__rf_[0-9a-fA-F]+)', '', stem)
    # Strip intermediate extensions like _jpg or _png before .rf
    stem = re.sub(r'(_jpg|_jpeg|_png)$', '', stem, flags=re.IGNORECASE)
    # Strip trailing numeric variants e.g. _1, _2, -1-, _0
    stem = re.sub(r'[-_]\d+[-_]?$', '', stem)
    return stem

duplication_records = []

for key, ds in resolved_datasets.items():
    audit = dataset_audits.get(key)
    if not audit or not audit['image_paths']:
        duplication_records.append({
            'Dataset': ds['name'],
            'Total Images': 0,
            'Unique Base IDs': 0,
            'Replication Ratio': 'unverified',
            'Estimated Augmentation': 'unverified',
            'Sample Base Groups': 'unverified'
        })
        continue
        
    # Filter to non-mask images to count distinct source scenes
    imgs = [p for p in audit['image_paths'] if '_mask' not in p.name.lower()]
    pool = imgs if len(imgs) > 0 else audit['image_paths']
    
    base_groups = defaultdict(list)
    for p in pool:
        b_id = extract_base_id(p.name)
        base_groups[b_id].append(p.name)
        
    total_imgs = len(pool)
    unique_bases = len(base_groups)
    ratio = total_imgs / unique_bases if unique_bases > 0 else 1.0
    
    # Find examples with multiple variants
    multi_variant_examples = [f"{b} ({len(v)} variants)" for b, v in base_groups.items() if len(v) > 1][:3]
    examples_str = ", ".join(multi_variant_examples) if multi_variant_examples else "No duplicate variants found"
    
    duplication_records.append({
        'Dataset': ds['name'],
        'Total Images': total_imgs,
        'Unique Base IDs': unique_bases,
        'Replication Ratio': f"{ratio:.2f}x",
        'Estimated Augmentation': f"{(ratio - 1.0)*100:.1f}% augmented" if ratio > 1.05 else "Original / Raw (No Roboflow duplicates)",
        'Sample Base Groups': examples_str
    })

df_duplication = pd.DataFrame(duplication_records)
print("\nDuplicate / Augmented Variant Summary:")
display(df_duplication[['Dataset', 'Total Images', 'Unique Base IDs', 'Replication Ratio', 'Estimated Augmentation']])

for r in duplication_records:
    print(f"\n[{r['Dataset']}] Base Group Samples:")
    print(f"  --> {r['Sample Base Groups']}")


In [ ]:
print("=" * 80)
print("DYNAMIC DATASET COMPARISON MATRIX (COMPUTED DIRECTLY FROM DATA)")
print("=" * 80)

matrix_rows = []

for key, ds in resolved_datasets.items():
    audit = dataset_audits.get(key)
    res_info = next((r for r in resolution_table_data if r['Dataset'] == ds['name']), {})
    dup_info = next((d for d in duplication_records if d['Dataset'] == ds['name']), {})
    
    if not audit:
        matrix_rows.append({
            'Dataset': ds['name'],
            'Owner': ds['owner'],
            'Total Files': 'unverified',
            'Image Count': 'unverified',
            'Label / Mask Count': 'unverified',
            'Extensions': 'unverified',
            'Verified Label Format': 'unverified',
            'Common Resolution': 'unverified',
            'Unique Scenes': 'unverified',
            'Segmentation Readiness': 'unverified'
        })
        continue
        
    # Determine verified label format
    if key in mask_dataset_results:
        verified_fmt = mask_dataset_results[key].get('verified_format', 'unverified')
        readiness = 'Verified: Direct Raster Mask (Ready for U-Net/FPN)'
    elif key == 'pothole':
        verified_fmt = pothole_results.get('verified_format', 'unverified')
        readiness = 'Verified: Polygon Vector (Ready via cv2.fillPoly)'
    elif key == 'rdd2022es':
        verified_fmt = rdd_results.get('verified_format', 'unverified')
        readiness = 'Unverified for pixel masks (Bounding boxes only)'
    else:
        verified_fmt = 'unverified'
        readiness = 'unverified'
        
    exts_str = ", ".join([f"{ext} ({cnt})" for ext, cnt in list(audit['extension_counts'].items())[:4]])
    
    matrix_rows.append({
        'Dataset': ds['name'],
        'Owner': ds['owner'],
        'Total Files': audit['total_files'],
        'Image Count': len(audit['image_paths']),
        'Label / Mask Count': len(audit['label_paths']),
        'Extensions': exts_str,
        'Verified Label Format': verified_fmt,
        'Common Resolution': res_info.get('Primary WxH', 'unverified'),
        'Unique Scenes': dup_info.get('Unique Base IDs', 'unverified'),
        'Segmentation Readiness': readiness
    })

df_matrix = pd.DataFrame(matrix_rows)
display(df_matrix)


## Observations & Key Questions

Review the computed matrices and visualizations above and record your findings:

1. **Image Resolutions & Variations**:
   - Are image resolutions standardized across datasets (e.g. $640 \times 640$) or variable (e.g. high-resolution phone/road camera captures)?
   - What image resize or letterbox target should be used for model input tensors?
2. **Label Formats**:
   - Which datasets provide ground truth masks with verified foreground pixel values?
   - How clean are the polygon rasterizations for the pothole dataset?
3. **Four Target Distress Classes**:
   - Confirm which distress categories are present in each dataset:
     - Longitudinal Cracks:
     - Transverse Cracks:
     - Alligator Cracks:
     - Potholes:
4. **Augmentation & Duplication Impact**:
   - What replication ratio was measured by the Base ID grouping?
   - Which datasets contain augmented duplicate views that must be kept in the same split to avoid data leakage between train and test?
5. **Next Architecture Decision**:
   - How should we unify binary crack masks, rasterized pothole masks, and multi-class RDD annotations into a coherent Pavement Condition Index (PCI) pipeline?
